In [1]:
!unzip /content/flight_data_2018_2024.csv.zip

Archive:  /content/flight_data_2018_2024.csv.zip
  inflating: flight_data_2018_2024.csv  


In [2]:
!pip install deltalake

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 114.6 MB/s eta 0:00:00


In [3]:
import polars as pl
from deltalake import DeltaTable, write_deltalake

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, mean_squared_error, mean_absolute_error, r2_score)
import xgboost as xgb
import warnings
from datetime import datetime
import time

warnings.filterwarnings('ignore')

# Polars + Datalake experiments

In [4]:
def optimize_zorder(table_path):
    dt = DeltaTable(table_path)
    dt.optimize.z_order(['route'])
    dt.optimize.compact()

def vacuum_old(table_path, retention_hours=24):
    dt = DeltaTable(table_path)
    dt.vacuum(
        retention_hours=retention_hours,
        enforce_retention_duration=False,
        dry_run=False
        )

In [5]:
def load_bronze_by_days(csv_path: str, delta_path: str, batch_size_days: int = 7):
    """
    Загружает CSV, разбивая на батчи по дням для имитации инкрементальной загрузки.
    batch_size_days: сколько дней в одном батче (например, 7 = недельными партициями)
    """
    # Читаем весь CSV
    full_df = pl.read_csv(csv_path)

    # Преобразуем FlightDate в дату
    df_with_date = full_df.with_columns(
        pl.col("FlightDate").str.strptime(pl.Date, "%Y-%m-%d").alias("Date")
    )

    # Получаем уникальные дни
    unique_days = df_with_date.select("Date").unique().sort("Date").to_series().to_list()

    # Разбиваем на батчи по неделям
    for i in range(0, len(unique_days), batch_size_days):
        batch_days = unique_days[i:i + batch_size_days]
        batch_df = df_with_date.filter(pl.col("Date").is_in(batch_days))

        if not DeltaTable.is_deltatable(delta_path):
            write_deltalake(delta_path, batch_df, mode="overwrite")
        else:
            write_deltalake(delta_path, batch_df, mode="append")


In [6]:
csv_path = '/content/flight_data_2018_2024.csv'
bronze_path = '/content/delta-files'
silver_path = '/content/silver-path'
gold_path = '/content/gold-path'

load_bronze_by_days(csv_path, bronze_path)

In [7]:
def create_silver(bronze_path: str, silver_path: str):
    bronze = pl.scan_delta(bronze_path)
    # Очистка и признаки
    clean_df = (
        bronze
        # отбираем только свершившиеся рейсы
        .filter(pl.col("Cancelled") == 0)
        # убираем аномальные значния признаков
        .filter(pl.col("DepTime")!="")
        .filter(pl.col("ArrTime")!="")
        .filter(pl.col("DepDelayMinutes").is_not_null())
        .filter(pl.col("ArrDelayMinutes").is_not_null())
        # создаем новые признаки
        .with_columns([
            pl.col("Date").dt.weekday().alias("day_of_week"),
            (pl.col("Origin") + pl.col("Dest")).alias("route"),
        ])
        .with_columns(
            pl.when(pl.col("DepTime")=="2400")
            .then(pl.lit(0))
            .otherwise(
                pl.col("DepTime").str.slice(0, 2).cast(pl.Int32)
            )
            .alias("dep_hour")
        )
        # выбираем только нужные колонки
        .select(["FlightDate", "Flight_Number_Marketing_Airline",
                 "ArrDelayMinutes", "DepDelayMinutes", "Distance",
                 "dep_hour", "day_of_week", "route",
                 "Year", "Origin", "Marketing_Airline_Network"])
    )
    try:
        # если таблица создана, то мы только переписываем добавляем
        # новые данные, не переписывая старые
        dt = DeltaTable(silver_path)

        predicate = "s.FlightDate = t.FlightDate AND s.Flight_Number_Marketing_Airline = t.Flight_Number_Marketing_Airline"
        dt.merge(
            clean_df.collect().to_arrow(),
            predicate=predicate,
            source_alias="s",
            target_alias="t"
        ).when_not_matched_insert_all().execute()

    except:
        write_deltalake(
            silver_path,
            clean_df.collect().to_arrow(),
            mode="append",
            partition_by=['FlightDate']
        )
    print('Query schema:')
    print(clean_df.explain(format='plain'))

In [8]:
create_silver(bronze_path, silver_path)

Query schema:
simple π 11/11 ["FlightDate", ... 10 other columns]
   WITH_COLUMNS:
   [col("Date").dt.weekday().alias("day_of_week"), col("Origin").str.concat_horizontal([col("Dest")]).alias("route"), when([(col("DepTime")) == ("2400")]).then(0).otherwise(col("DepTime").str.slice([dyn int: 0, dyn int: 2]).strict_cast(Int32)).alias("dep_hour")] 
    Parquet SCAN [/content/delta-files/part-00000-7f1f5275-2c27-4f4a-83ab-430ab392313e-c000.snappy.parquet, ... 4 other sources]
    PROJECT 13/121 COLUMNS
    SELECTION: [([([([([(col("Cancelled")) == (0.0)]) & (col("DepDelayMinutes").is_not_null())]) & ([(col("ArrTime")) != ("")])]) & ([(col("DepTime")) != ("")])]) & (col("ArrDelayMinutes").is_not_null())]


In [ ]:
t_lazy = time.perf_counter()

pl.scan_delta(silver_path)

all_lazy = time.perf_counter() - t_lazy

print(f'Lazy mode on: {all_lazy}')

t_start = time.perf_counter()

pl.read_delta(silver_path)

all_time = time.perf_counter() - t_start

print(f'Lazy mode off: {all_time}')

Lazy mode on: 0.007072929000059958
Lazy mode off: 0.05582277199982855


Видим, что Lazy mode помогает нам увеличить скорость почти в 10 раз

In [ ]:
t_lazy = time.perf_counter()

pl.scan_delta(silver_path)

all_lazy = time.perf_counter() - t_lazy

print(f'Optimizations off: {all_lazy}')

optimize_zorder(silver_path)
vacuum_old(silver_path, retention_hours=24)

t_lazy_opt = time.perf_counter()

pl.scan_delta(silver_path)

all_lazy_opt = time.perf_counter() - t_lazy_opt

print(f'Optimizations on: {all_lazy}')

Optimizations off: 0.008244805000003907
Optimizations on: 0.008244805000003907


In [9]:
def build_aggregates(silver_path: str, gold_agg_path: str):
    df = pl.scan_delta(silver_path)

    aggregations = [
        {
            "group_by": ["route", "dep_hour"],
            "output_path": f"{gold_agg_path}/agg_route"
        },
        {
            "group_by": ["Marketing_Airline_Network", "Flight_Number_Marketing_Airline"],
            "output_path": f"{gold_agg_path}/agg_airline"
        }
    ]

    for agg_config in aggregations:
        result = (
            df.group_by(agg_config["group_by"])
            .agg(
                pl.col("ArrDelayMinutes").mean().alias("avg_arr_delay"),
                pl.col("DepDelayMinutes").mean().alias("avg_dep_delay")
            )
            .collect()
        )
        result.write_delta(agg_config["output_path"], mode="overwrite")

def build_feature_table(silver_path: str, gold_feat_path: str, n: int):
    path = gold_feat_path + '/ml_data'
    df = pl.scan_delta(silver_path)
    encoded = (
        df.select(
            ["ArrDelayMinutes", "Distance", "dep_hour",
             "day_of_week", "Origin", "Flight_Number_Marketing_Airline"]
            )
        .with_columns(
            pl.when(pl.col('ArrDelayMinutes') >= n)
            .then(pl.lit(1))
            .otherwise(pl.lit(0))
            .alias('is_delayed')
            )
        )
    encoded.collect().write_delta(path, mode="overwrite")

In [10]:
build_aggregates(silver_path, gold_path)

In [21]:
build_feature_table(silver_path, gold_path, 1)

# ML experiments

## Classification

In [23]:
data = pl.scan_delta('/content/gold-path/ml_data').collect().to_pandas().sample(100_000)
y_reg = data['ArrDelayMinutes']
y_cls = data['is_delayed']

X_reg = data.drop(columns=['is_delayed', 'ArrDelayMinutes'])
X_cls = data.drop(columns=['is_delayed', 'ArrDelayMinutes'])

In [ ]:
categorical_cols = X_cls.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = X_cls.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"Categorical columns: {categorical_cols}")
print(f"Numerical columns: {numerical_cols}")

Categorical columns: ['Origin']
Numerical columns: ['Distance', 'Flight_Number_Marketing_Airline']


In [ ]:
categorical_cols = X_reg.select_dtypes(include=['object', 'category']).columns.tolist()

X_reg.select_dtypes(include=['int64', 'float64']).columns.tolist()

['Distance', 'Flight_Number_Marketing_Airline']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)

### Logisitic regression

In [ ]:
preprocessor_cls = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ])

classification_results = {}

logreg_pipeline = Pipeline([
    ('preprocessor', preprocessor_cls),
    ('classifier', LogisticRegression(random_state=42))
])

param_grid = {
    "classifier__C": [1, 10, 100],
    "classifier__max_iter": [100, 500, 700],
    "classifier__class_weight": ['balanced']
}

logreg_grid = GridSearchCV(
    logreg_pipeline,
    param_grid,
    cv=5,
    scoring='neg_log_loss',
    verbose=1
)

logreg_grid.fit(X_train, y_train)

print(f"Best CV log_loss: {np.sqrt(-logreg_grid.best_score_):.2f}")

best_logreg = logreg_grid.best_estimator_
y_pred_logreg = best_logreg.predict(X_test)
y_pred_proba_logreg = best_logreg.predict_proba(X_test)[:, 1]

classification_results['Logistic Regression'] = {
    'accuracy': accuracy_score(y_test, y_pred_logreg),
    'precision': precision_score(y_test, y_pred_logreg),
    'recall': recall_score(y_test, y_pred_logreg),
    'f1': f1_score(y_test, y_pred_logreg),
    'roc_auc': roc_auc_score(y_test, y_pred_proba_logreg)
}

print(f"Accuracy: {classification_results['Logistic Regression']['accuracy']:.4f}")
print(f"F1-Score: {classification_results['Logistic Regression']['f1']:.4f}")
print(f"ROC-AUC: {classification_results['Logistic Regression']['roc_auc']:.4f}")

Fitting 5 folds for each of 9 candidates, totalling 45 fits
Best CV log_loss: 0.83
Accuracy: 0.5385
F1-Score: 0.3642
ROC-AUC: 0.5613


### Random forest

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

rf_pipeline = Pipeline([
    ('preprocessor', preprocessor_cls),
    ('classifier', RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight='balanced'
    ))
])

param_grid_rf = {
    "classifier__n_estimators": [50, 100, 150],
    "classifier__max_depth": [4, 6, 10],
}

rf_grid = GridSearchCV(
    rf_pipeline,
    param_grid_rf,
    cv=5,
    scoring='neg_log_loss',
    verbose=1
    )

rf_grid.fit(X_train, y_train)

best_rf = rf_grid.best_estimator_

y_pred_rf = best_rf.predict(X_test)
y_pred_proba_rf = best_rf.predict_proba(X_test)[:, 1]

classification_results['Random Forest'] = {
    'accuracy': accuracy_score(y_test, y_pred_rf),
    'precision': precision_score(y_test, y_pred_rf),
    'recall': recall_score(y_test, y_pred_rf),
    'f1': f1_score(y_test, y_pred_rf),
    'roc_auc': roc_auc_score(y_test, y_pred_proba_rf)
}

print(f"Best CV log_loss: {np.sqrt(-rf_grid.best_score_):.2f}")
print(f"Accuracy: {classification_results['Random Forest']['accuracy']:.4f}")
print(f"F1-Score: {classification_results['Random Forest']['f1']:.4f}")
print(f"ROC-AUC: {classification_results['Random Forest']['roc_auc']:.4f}")

Fitting 5 folds for each of 9 candidates, totalling 45 fits
Best CV log_loss: 0.83
Accuracy: 0.6453
F1-Score: 0.3158
ROC-AUC: 0.5655


### XGBoost

In [ ]:
import xgboost as xgb
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor_cls),
    ('classifier', xgb.XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        n_jobs=-1,
        eval_metric='logloss',
        use_label_encoder=False
    ))
])

param_grid_xgb = {
    "classifier__n_estimators": [100, 150, 200, 300],
    "classifier__max_depth": [3, 6, 9],
}

xgb_grid = GridSearchCV(
    xgb_pipeline,
    param_grid_xgb,
    cv=5,
    scoring='neg_log_loss',
    verbose=1
)

xgb_grid.fit(X_train, y_train)

best_xgb = xgb_grid.best_estimator_

y_pred_xgb = best_xgb.predict(X_test)
y_pred_proba_xgb = best_xgb.predict_proba(X_test)[:, 1]

classification_results['XGBoost'] = {
    'accuracy': accuracy_score(y_test, y_pred_xgb),
    'precision': precision_score(y_test, y_pred_xgb),
    'recall': recall_score(y_test, y_pred_xgb),
    'f1': f1_score(y_test, y_pred_xgb),
    'roc_auc': roc_auc_score(y_test, y_pred_proba_xgb)
}

print(f"Best CV log_loss: {np.sqrt(-xgb_grid.best_score_):.2f}")
print(f"Accuracy: {classification_results['XGBoost']['accuracy']:.4f}")
print(f"F1-Score: {classification_results['XGBoost']['f1']:.4f}")
print(f"ROC-AUC: {classification_results['XGBoost']['roc_auc']:.4f}")

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best CV log_loss: 0.74
Accuracy: 0.7607
F1-Score: 0.0071
ROC-AUC: 0.5662


### Сравнение классификаторов

In [ ]:
results_df = pd.DataFrame(classification_results).T
print(results_df.round(4))

best_clf = results_df['f1'].idxmax()
print(f"\nBest classifier based on F1-score: {best_clf}")

                     accuracy  precision  recall      f1  roc_auc
Logistic Regression    0.5385     0.2715  0.5533  0.3642   0.5613
Random Forest          0.6454     0.2929  0.3425  0.3158   0.5655
XGBoost                0.7607     0.4048  0.0036  0.0071   0.5662

Best classifier based on F1-score: Logistic Regression


Видим, что по итогу обучения лучше всех справляется логистическая регрессия. Xgboost, несмотря на высокий accuracy, стал константной моделью. Случайный лес больше имеет большую обобщающую способность по сравнению с предыдущей моделью, однако пропускает больше положительных классов по сравнению с логистической регрессией.

Ввиду описанных результатов, мы выберем логистическую регрессию.

## Regression

In [ ]:
# Предобработка для регрессии с OneHotEncoder
categorical_cols_reg = X_reg.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols_reg = X_reg.select_dtypes(include=['int64', 'float64']).columns.tolist()

preprocessor_reg = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols_reg),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols_reg)
    ])

# Разделение данных для регрессии
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

### Liner regression + l2 regularization

In [ ]:
param_grid = {'regressor__alpha': [0.1, 1.0, 10.0, 50.0, 100.0]}

ridge_pipeline = Pipeline([
    ('preprocessor', preprocessor_reg),
     ('regressor', Ridge(random_state=42))
     ])

ridge_grid = GridSearchCV(
    ridge_pipeline,
    param_grid,
    cv=2,
    scoring='neg_mean_squared_error',
    verbose=1
)

ridge_grid.fit(X_train_reg, y_train_reg)

print(f"\nBest alpha: {ridge_grid.best_params_['regressor__alpha']}")
print(f"Best CV RMSE: {np.sqrt(-ridge_grid.best_score_):.2f}")

# Лучшая модель
best_ridge = ridge_grid.best_estimator_
y_pred_best_ridge = best_ridge.predict(X_test_reg)

best_ridge_metrics = {
    'RMSE': np.sqrt(mean_squared_error(y_test_reg, y_pred_best_ridge)),
    'MAE': mean_absolute_error(y_test_reg, y_pred_best_ridge),
    'R2': r2_score(y_test_reg, y_pred_best_ridge)
}

print(f"\nTest set performance with best alpha:")
print(f"RMSE: {best_ridge_metrics['RMSE']:.2f}")
print(f"MAE: {best_ridge_metrics['MAE']:.2f}")
print(f"R2: {best_ridge_metrics['R2']:.4f}")

Fitting 2 folds for each of 5 candidates, totalling 10 fits

Best alpha: 50.0
Best CV RMSE: 65.21 minutes

Test set performance with best alpha:
RMSE: 63.89 minutes
MAE: 27.56 minutes
R2: 0.0082


### Random forest

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("Random Forest Regressor")

rf_reg_pipeline = Pipeline([
    ('preprocessor', preprocessor_reg),
    ('regressor', RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ))
])

param_grid_rf_reg = {
    "regressor__n_estimators": [50, 100],
    "regressor__max_depth": [3, 6, 9],
}

rf_reg_grid = GridSearchCV(
    rf_reg_pipeline,
    param_grid_rf_reg,
    cv=3,
    scoring='neg_root_mean_squared_error',
    verbose=1
)

rf_reg_grid.fit(X_train_reg, y_train_reg)

best_rf_reg = rf_reg_grid.best_estimator_

y_pred_rf_reg = best_rf_reg.predict(X_test_reg)

# Метрики
rf_reg_metrics = {
    'RMSE': np.sqrt(mean_squared_error(y_test_reg, y_pred_rf_reg)),
    'MAE': mean_absolute_error(y_test_reg, y_pred_rf_reg),
    'R2': r2_score(y_test_reg, y_pred_rf_reg)
}

print(f"Best CV RMSE: {np.sqrt(-rf_reg_grid.best_score_):.2f}")
print(f"RMSE: {rf_reg_metrics['RMSE']:.2f}")
print(f"MAE: {rf_reg_metrics['MAE']:.2f}")
print(f"R2: {rf_reg_metrics['R2']:.4f}")

Random Forest Regressor
Fitting 3 folds for each of 6 candidates, totalling 18 fits
Best CV RMSE: 8.07
RMSE: 63.67
MAE: 27.60
R2: 0.0095


Обучив две модели для задачи регрессии, можно заметить, что они дают примерно одинаковые результаты по выбранным нами метрикам. Однако линейная регрессия обучалась намного быстрее, чем случайный лес, поэтому мы выберем её.